# AOL Evaluation - Random Edits (T = 1)

Random substitution, insertion, and deletion are evaluated using the same saved pivots and `GT` labels as SPOT.

This notebook sweeps the official AOL threshold grid from 1.00 to 3.00 in increments of 0.01, selects operating points using the FPR-bin rule at target FPR 0.05, and measures runtime at each selected threshold.

**Input:** `post_edit_random_T1.zip`  
**Output:** `evaluation_random_AOL_T1.zip`


In [ ]:
%pip -q install pandas pybind11 setuptools


In [ ]:
import glob
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
def locate_zip(filename: str) -> Path:
    candidates = [Path('/content') / filename, Path('/mnt/data') / filename, Path.cwd() / filename]
    for path in candidates:
        if path.exists() and zipfile.is_zipfile(path):
            return path
    try:
        from google.colab import files
        uploaded = files.upload()
        for uploaded_name in uploaded:
            path = Path('/content') / uploaded_name
            if path.suffix.lower() == '.zip' and zipfile.is_zipfile(path):
                return path
    except Exception:
        pass
    raise FileNotFoundError(f'Could not find {filename}.')


def load_dataset_zip(filename: str, workdir_name: str):
    zip_path = locate_zip(filename)
    root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    workdir = root / workdir_name
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(workdir)

    dataset_paths = list(workdir.rglob('dataset.npz'))
    meta_paths = list(workdir.rglob('meta.json'))
    if len(dataset_paths) != 1 or len(meta_paths) != 1:
        raise FileNotFoundError('The input ZIP must contain one dataset.npz and one meta.json.')
    return (
        np.load(dataset_paths[0], allow_pickle=False),
        json.loads(meta_paths[0].read_text(encoding='utf-8')),
        zip_path.stem,
    )


def zip_output_folder(folder: Path, zip_name: str) -> Path:
    root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    zip_path = root / zip_name
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=folder)
    print('Saved:', zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
    return zip_path


In [ ]:
# -----------------------
# Repo C++ core (cpp_src/aligator.cpp) embedded verbatim
# -----------------------
# NOTE: This is embedded exactly from llm-watermark-location-main/cpp_src/aligator.cpp

ALIGATOR_CPP = r"""#include<pybind11/pybind11.h>
#include<pybind11/stl.h>
#include<pybind11/numpy.h>
#include<cmath>
#include<vector>
#include<iostream>

using namespace std;
namespace py = pybind11;
double prev_pred = 0;

struct expert {
    double prediction;
    double loss;
    int count;
    double weight;

    expert() {
        prediction = 0;
        loss = 0;
        count = 0;
        weight = 0;
    }
};

int init_experts(std::vector<std::vector<expert> > &pool, int n) {
    int count = 0;
    for (int k = 0; k <= floor(log2(n)); k++) {
        int stop = ((n + 1) >> k) - 1;
        if (stop < 1)
            break;
        std::vector<expert> elist;
        for (int i = 1; i <= stop; i++) {
            expert e;
            elist.push_back(e);
            if( k > 4 ) count++;
        }
        pool.push_back(elist);
    }
    return count;
}

void get_awake_set(std::vector<int> &index, int t, int n) {
    // int j = 0;
    for (int k = 0; k <= floor(log2(t)); k++) {
        int i = (t >> k);
        if (((i + 1) << k) - 1 > n  || k<=4)
            index.push_back(-1);
        // j++;
        else
            index.push_back(i);
    }
    // if(j == floor(log2(t))+1){
    //     std::cout<<"failed! "<<t<<std::endl;
    // }
}

double get_forecast(std::vector<int> &awake_set,
                    std::vector<std::vector<expert> > &pool,
                    double &normalizer, int pool_size) {
    double output = 0;
    normalizer = 0;
    int i;
    for (int k = 0; k < awake_set.size(); k++) {
        if (awake_set[k] == -1) continue;
        i = awake_set[k] - 1;
        if (pool[k][i].weight == 0) {
            pool[k][i].weight = 1.0 / pool_size;
            // added to reduce jittery output for isotonic case
            pool[k][i].prediction = prev_pred;
        }
        output = output + (pool[k][i].weight * pool[k][i].prediction);
        normalizer = normalizer + pool[k][i].weight;
    }
    return output / normalizer;
}

void compute_losses(std::vector<int> &awake_set,
                    std::vector<std::vector<expert> > &pool,
                    std::vector<double> &losses, double y,
                    double B, int n, double sigma, double delta) {
    int i;
    double norm = 2 * (B + sigma * sqrt(log(2 * n / delta))) * (B + sigma * sqrt(log(2 * n / delta)));
    // double norm = sigma;

    for (int k = 0; k < awake_set.size(); k++) {
        if (awake_set[k] == -1) {
            losses.push_back(-1);
        } else {
            i = awake_set[k] - 1;
            double loss = (y - pool[k][i].prediction) * (y - pool[k][i].prediction) / norm;
            losses.push_back(loss);
        }
    }
}

void update_weights_and_predictions(std::vector<int> &awake_set,
                                    std::vector<std::vector<expert> > &pool,
                                    std::vector<double> &losses,
                                    double normalizer, double y) {
    double norm = 0;
    int i;
    // compute new normalizer
    for (int k = 0; k < awake_set.size(); k++) {
        if (awake_set[k] == -1) continue;
        i = awake_set[k] - 1;
        norm = norm + pool[k][i].weight * exp(-losses[k]);
    }
    // update weights and predictions
    for (int k = 0; k < awake_set.size(); k++) {
        if (awake_set[k] == -1) continue;
        i = awake_set[k] - 1;
        pool[k][i].weight = pool[k][i].weight * exp(-losses[k]) * normalizer / norm;
        pool[k][i].prediction = ((pool[k][i].prediction * pool[k][i].count) + y) / (pool[k][i].count + 1);
        pool[k][i].count = pool[k][i].count + 1;
    }
}

std::vector<double> run_aligator(int n, std::vector<double> y,
                                 std::vector<int> index,
                                 double sigma,
                                 double B, double delta) {
    prev_pred = 0;
    std::vector<double> estimates(n);
    std::vector<std::vector<expert> > pool;
    int pool_size = init_experts(pool, n);
    for (int t = 0; t < n; t++) {
        double normalizer = 0;
        std::vector<int> awake_set;
        int idx = index[t];
        double y_curr = y[idx];
        get_awake_set(awake_set, idx + 1, n);
        double output = get_forecast(awake_set, pool, normalizer, pool_size);
        estimates[idx] = output;
        // if(output == 0)
        //     std::cout<<"no predict: "<<idx<<std::endl;
        // if(idx<10 ){
        //     std::cout<<"idx: "<<idx<<" output: "<<output<<std::endl;
        //     for(int k = 0; k < awake_set.size(); k++){
        //         std::cout<< awake_set[k]<<' ';
        //     }
        //     std::cout<<std::endl;
        // }
        std::vector<double> losses;
        compute_losses(awake_set, pool, losses, y_curr, B, n, sigma, delta);
        update_weights_and_predictions(awake_set, pool, losses, normalizer, y_curr);
        prev_pred = y_curr;
    }
    return estimates;
}


PYBIND11_MODULE(aligator, m) {
    m.doc() = "pybind11 aligator plugin"; // optional module docstring

    m.def("run_aligator", [](int n, std::vector<double> y, std::vector<int> index, \
			     double sigma, double B, double delta) -> py::array {
	    auto v = run_aligator(n,y,index,sigma,B,delta);
	return py::array(v.size(), v.data());
	  },py::arg("n"), py::arg("index"), py::arg("y"), py::arg("sigma"),	\
	  py::arg("B"), py::arg("delta"));
}

/*
Linux:
c++ -O3 -Wall -shared -std=c++11 -fPIC `python3 -m pybind11 --includes` aligator.cpp -o aligator`python3-config --extension-suffix`
Mac:
c++ -O3 -Wall -shared -std=c++11 -undefined dynamic_lookup `python3 -m pybind11 --includes` aligator.cpp -o aligator`python3-config --extension-suffix`

 */
"""

import sys, subprocess, importlib.util

def _ensure_pybind11():
    try:
        import pybind11  # noqa
        return
    except Exception:
        pass
    print("Installing pybind11...")
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pybind11"])

def build_aligator_inline(build_root=None):
    if build_root is None:
        base = "/content" if os.path.isdir("/content") else "/mnt/data"
        build_root = os.path.join(base, "_aol_inline_build")
    """Build the pybind11 extension module `aligator` from embedded C++ source."""
    _ensure_pybind11()
    import pybind11
    from setuptools import setup, Extension
    from setuptools.command.build_ext import build_ext

    build_root = os.path.abspath(build_root)
    src_dir = os.path.join(build_root, "cpp_src")
    os.makedirs(src_dir, exist_ok=True)
    cpp_path = os.path.join(src_dir, "aligator.cpp")
    with open(cpp_path, "w", encoding="utf-8") as f:
        f.write(ALIGATOR_CPP)

    ext_modules = [
        Extension(
            name="aligator",
            sources=[cpp_path],
            include_dirs=[pybind11.get_include()],
            language="c++",
            extra_compile_args=["-O3"],
        )
    ]

    class BuildExt(build_ext):
        def build_extensions(self):
            super().build_extensions()

    build_temp = os.path.join(build_root, "_build")
    os.makedirs(build_temp, exist_ok=True)

    cwd = os.getcwd()
    os.chdir(build_root)
    try:
        setup(
            name="aligator",
            ext_modules=ext_modules,
            cmdclass={"build_ext": BuildExt},
            script_args=["build_ext", "--inplace", "--build-temp", build_temp],
            zip_safe=False,
        )
    finally:
        os.chdir(cwd)

    # import built module
    so_glob = glob.glob(os.path.join(build_root, "aligator*.so"))
    if not so_glob:
        raise RuntimeError(f"Build succeeded but no aligator*.so found in {build_root}")
    so_path = so_glob[0]
    spec = importlib.util.spec_from_file_location("aligator", so_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)  # type: ignore
    return mod

aligator = build_aligator_inline()

def aol_scores_from_pivots(y, eps=1e-12):
    """AOL paper's Gumbel token score uses s = log(1/(1-r)) = -log(1-r)."""
    y = np.asarray(y, dtype=float)
    y = np.clip(y, 0.0, 1.0 - eps)
    return -np.log(1.0 - y)

def aligator_smooth_like_repo(y, B=1.0, delta=1e-5):
    """Compute the Aligator-smoothed score sequence (repo-faithful), independent of threshold.

    This is the expensive part. Once you have `smoothed`, you can sweep many thresholds cheaply via:
        detect_idx = np.where(smoothed > thr)[0]
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n == 0:
        return np.array([])

    step = max(int(n / 30), 1)

    # Precompute index lists once (minor speedup)
    idx_fwd_list = list(range(n))
    idx_rev_list = idx_fwd_list[::-1]

    res = []
    yy = y.copy()
    for shift in range(0, n, step):
        # Convert to Python list once per shift (reused for fwd/rev)
        yy_list = yy.tolist()

        alig1 = aligator.run_aligator(n, yy_list, idx_fwd_list, 0.0, B, delta)
        alig2 = aligator.run_aligator(n, yy_list, idx_rev_list, 0.0, B, delta)

        alig = np.nanmean(np.array([np.asarray(alig1), np.asarray(alig2)]), axis=0)

        # Undo the circular shift so scores align with original token positions
        alig = np.concatenate((alig[n - shift :], alig[0 : n - shift]))
        res.append(alig)

        # Rotate input scores for the next shift window
        yy = np.concatenate((yy[step:], yy[0:step]))

    smoothed = np.nanmean(np.array(res), axis=0)
    return smoothed


In [ ]:
def token_metrics(prediction: np.ndarray, ground_truth: np.ndarray) -> dict:
    prediction = np.asarray(prediction, dtype=bool)
    ground_truth = np.asarray(ground_truth, dtype=bool)
    tp = int(np.sum(prediction & ground_truth))
    fp = int(np.sum(prediction & ~ground_truth))
    tn = int(np.sum(~prediction & ~ground_truth))
    fn = int(np.sum(~prediction & ground_truth))
    return {
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'true_count': tp + fn,
        'IoU': float(tp / (tp + fp + fn + 1e-12)),
        'FPR': float(fp / (fp + tn + 1e-12)),
    }


def summarize_metrics(metrics: pd.DataFrame, parameter_column: str) -> pd.DataFrame:
    group_columns = ['method', 'case', 'level_index', 'edit_level', parameter_column]
    grouped = metrics.groupby(group_columns, dropna=False, as_index=False)
    summary = grouped.agg(
        TP_mean=('TP', 'mean'),
        TP_sum=('TP', 'sum'),
        true_sum=('true_count', 'sum'),
        IoU=('IoU', 'mean'),
        FPR=('FPR', 'mean'),
        n_documents=('document_index', 'count'),
    )
    summary['TPR'] = summary['TP_sum'] / summary['true_sum'].replace(0, np.nan)
    return summary


def select_best_with_fpr_bin(summary: pd.DataFrame, metric: str,
                              parameter_column: str,
                              target_fpr: float = 0.05,
                              bin_width: float = 0.01) -> pd.DataFrame:
    bin_uppers = np.round(np.arange(bin_width, target_fpr + 1e-12, bin_width), 2)
    records = []
    group_columns = ['method', 'case', 'level_index', 'edit_level']

    for group_values, group in summary.groupby(group_columns, dropna=False, sort=True):
        chosen = None
        used_upper = np.nan
        fallback_steps = np.nan
        for index in range(len(bin_uppers) - 1, -1, -1):
            upper = float(bin_uppers[index])
            lower = round(upper - bin_width, 2)
            if lower <= 0:
                mask = (group['FPR'] >= lower - 1e-12) & (group['FPR'] <= upper + 1e-12)
            else:
                mask = (group['FPR'] > lower - 1e-12) & (group['FPR'] <= upper + 1e-12)
            candidates = group[mask]
            if not candidates.empty:
                candidates = candidates.sort_values(
                    [metric, 'FPR', parameter_column],
                    ascending=[False, False, True],
                )
                chosen = candidates.iloc[0]
                used_upper = upper
                fallback_steps = len(bin_uppers) - 1 - index
                break

        record = dict(zip(group_columns, group_values))
        record.update({
            'selection_metric': metric,
            'target_FPR': target_fpr,
            'used_bin_lower': used_upper - bin_width if np.isfinite(used_upper) else np.nan,
            'used_bin_upper': used_upper,
            'fallback_steps': fallback_steps,
            parameter_column: np.nan,
            'selected_FPR': np.nan,
            'selected_IoU': np.nan,
            'selected_TPR': np.nan,
        })
        if chosen is not None:
            record.update({
                parameter_column: float(chosen[parameter_column]),
                'selected_FPR': float(chosen['FPR']),
                'selected_IoU': float(chosen['IoU']),
                'selected_TPR': float(chosen['TPR']),
            })
        records.append(record)

    return pd.DataFrame(records)


## Configuration


In [ ]:
INPUT_ZIP_NAME = 'post_edit_random_T1.zip'
OUTPUT_ZIP_NAME = 'evaluation_random_AOL_T1.zip'

AOL_THRESHOLDS = [round(float(value), 2) for value in np.arange(1.00, 3.00 + 1e-12, 0.01)]
TARGET_FPR = 0.05


In [ ]:
data, meta, input_stem = load_dataset_zip(INPUT_ZIP_NAME, 'post_edit_random_T1_extracted')
required = {
    'Ys', 'S1',
    'Ys_post_sub_repo', 'GT_sub_repo',
    'Ys_post_ist_repo', 'GT_ist_repo',
    'Ys_post_dlt_repo', 'GT_dlt_repo',
}
missing = required.difference(data.files)
if missing:
    raise KeyError(f'Missing arrays: {sorted(missing)}')
if float(meta.get('temp', 1.0)) != 1.0:
    raise ValueError('This notebook is configured for T = 1.')

root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
output_folder = root / 'evaluation_random_AOL_T1'
if output_folder.exists():
    shutil.rmtree(output_folder)
output_folder.mkdir(parents=True, exist_ok=True)

edit_rates = np.asarray(meta.get('edit_rates', np.arange(0.0, 0.50 + 1e-12, 0.05)), dtype=float)
case_arrays = {
    'random_substitution': ('Ys_post_sub_repo', 'GT_sub_repo'),
    'random_insertion': ('Ys_post_ist_repo', 'GT_ist_repo'),
    'random_deletion': ('Ys_post_dlt_repo', 'GT_dlt_repo'),
}

samples = []
for case, (pivot_key, gt_key) in case_arrays.items():
    pivots = np.asarray(data[pivot_key])
    ground_truth = np.asarray(data[gt_key]).astype(bool)
    if pivots.shape != ground_truth.shape:
        raise ValueError(f'Shape mismatch for {case}: {pivots.shape} vs {ground_truth.shape}')
    for level_index, edit_rate in enumerate(edit_rates):
        for document_index in range(pivots.shape[1]):
            Y = np.asarray(pivots[level_index, document_index], dtype=float).reshape(-1)
            GT = np.asarray(ground_truth[level_index, document_index], dtype=bool).reshape(-1)
            keep = np.isfinite(Y)
            samples.append({
                'case': case,
                'level_index': int(level_index),
                'edit_level': float(edit_rate),
                'document_index': int(document_index),
                'Y': Y[keep],
                'GT': GT[keep],
            })
print('Samples:', len(samples))


## Evaluate, select operating points, and measure runtime


In [ ]:
rows = []
smoothed_cache = {}
for sample in samples:
    scores = aol_scores_from_pivots(sample['Y'])
    smoothed = aligator_smooth_like_repo(scores)
    cache_key = (sample['case'], sample['level_index'], sample['edit_level'], sample['document_index'])
    smoothed_cache[cache_key] = smoothed

    for threshold in AOL_THRESHOLDS:
        prediction = smoothed > threshold
        rows.append({
            'method': 'AOL',
            'case': sample['case'],
            'level_index': sample['level_index'],
            'edit_level': sample['edit_level'],
            'document_index': sample['document_index'],
            'threshold': float(threshold),
            **token_metrics(prediction, sample['GT']),
        })

metrics = pd.DataFrame(rows)
summary = summarize_metrics(metrics, 'threshold')
metrics.to_csv(output_folder / 'metrics.csv', index=False)
summary.to_csv(output_folder / 'metrics_summary.csv', index=False)

selected = pd.concat([
    select_best_with_fpr_bin(summary, 'IoU', 'threshold', TARGET_FPR),
    select_best_with_fpr_bin(summary, 'TPR', 'threshold', TARGET_FPR),
], ignore_index=True)
selected.to_csv(output_folder / 'selected_parameters.csv', index=False)
display(selected)

samples_by_group = {}
for sample in samples:
    key = (sample['case'], sample['level_index'])
    samples_by_group.setdefault(key, []).append(sample)

runtime_rows = []
for row in selected.itertuples(index=False):
    threshold = float(row.threshold)
    if not np.isfinite(threshold):
        continue
    group_samples = samples_by_group[(row.case, row.level_index)]

    def run_one(sample):
        scores = aol_scores_from_pivots(sample['Y'])
        return aligator_smooth_like_repo(scores) > threshold

    run_one(group_samples[0])
    started = time.perf_counter()
    for sample in group_samples:
        run_one(sample)
    elapsed = time.perf_counter() - started

    runtime_rows.append({
        'method': 'AOL',
        'case': row.case,
        'level_index': row.level_index,
        'edit_level': row.edit_level,
        'selection_metric': row.selection_metric,
        'threshold': threshold,
        'runtime_seconds_total': elapsed,
        'runtime_seconds_per_document': elapsed / len(group_samples),
        'n_documents': len(group_samples),
    })

runtime_columns = [
    'method', 'case', 'level_index', 'edit_level', 'selection_metric',
    'threshold', 'runtime_seconds_total', 'runtime_seconds_per_document', 'n_documents',
]
runtime_by_level = pd.DataFrame(runtime_rows, columns=runtime_columns)
runtime_by_level.to_csv(output_folder / 'runtime_by_level.csv', index=False)
display(runtime_by_level)

metadata = {
    'input_dataset': INPUT_ZIP_NAME,
    'temperature': 1.0,
    'method': 'AOL',
    'thresholds': AOL_THRESHOLDS,
    'target_FPR': TARGET_FPR,
    'selection_rule': 'Use the highest nonempty 0.01-wide FPR bin not exceeding the target, then maximize IoU or TPR within that bin.',
}
(output_folder / 'summary.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
zip_output_folder(output_folder, OUTPUT_ZIP_NAME)
